In [3]:
from collections import Counter

# 1. 텍스트 데이터 준비
text = "사과 바나나 사과 딸기 사과 바나나"

# 2. 토큰화 (공백 기준)
tokens = text.split()

# 3. 빈도수 계산
vocab = Counter(tokens)
print("단어별 빈도수:", vocab)

# 4. 빈도수가 높은 순서대로 정렬
sorted_vocab = sorted(vocab.items(), key=lambda x: x[1], reverse=True)
print(f'sorted_vocab : {sorted_vocab}')

# 5. 고유 정수 부여 (1부터 시작)
word_to_index = {}
for i, (word, frequency) in enumerate(sorted_vocab, start=1):
    word_to_index[word] = i

print("단어별 부여된 정수:", word_to_index)

# 6. 정수 인코딩 실행
encoded_text = [word_to_index[word] for word in tokens]
print("\n원본 텍스트:", tokens)
print("정수 인코딩:", encoded_text)

단어별 빈도수: Counter({'사과': 3, '바나나': 2, '딸기': 1})
sorted_vocab : [('사과', 3), ('바나나', 2), ('딸기', 1)]
단어별 부여된 정수: {'사과': 1, '바나나': 2, '딸기': 3}

원본 텍스트: ['사과', '바나나', '사과', '딸기', '사과', '바나나']
정수 인코딩: [1, 2, 1, 3, 1, 2]


In [2]:
import torch
from collections import Counter

# 1. 텍스트 데이터 준비
sentences = [
    "나는 학교에 간다",
    "나는 집에도 가고 학교에도 간다",
    "학교는 즐겁다"
]

# 2. 토큰화 (공백 기준)
tokenized_sentences = [sent.split() for sent in sentences]

# 3. 단어 빈도수 계산
tokens_all = [token for sent in tokenized_sentences for token in sent]
print(tokens_all)

vocab_counts = Counter(tokens_all)

# 4. 단어 집합(Vocabulary) 생성 (스페셜 토큰 포함)
# <PAD>: 길이를 맞추기 위한 패딩 (0)
# <UNK>: 단어장에 없는 단어 (1)
word_to_idx = {"<PAD>": 0, "<UNK>": 1}

# 빈도순 정렬 후 인덱스 부여
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

print("단어 집합(Vocab):", word_to_idx)

# 5. 정수 인코딩 (텍스트 -> 정수 리스트)
def encode(sentence_tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in sentence_tokens]

encoded_list = [encode(sent, word_to_idx) for sent in tokenized_sentences]
print(f'encoded_list : {encoded_list}')

# 6. PyTorch Tensor로 변환
# (실제 입력 시에는 문장 길이를 맞추는 Padding 작업이 함께 들어갑니다)
tensor_data = [torch.tensor(seq, dtype=torch.long) for seq in encoded_list]
print(f'tensor_data : {tensor_data}')

print("\n--- 결과 ---")
for sent, tensor in zip(sentences, tensor_data):
    print(f"원문: {sent}")
    print(f"Tensor: {tensor}\n")

['나는', '학교에', '간다', '나는', '집에도', '가고', '학교에도', '간다', '학교는', '즐겁다']
단어 집합(Vocab): {'<PAD>': 0, '<UNK>': 1, '나는': 2, '간다': 3, '학교에': 4, '집에도': 5, '가고': 6, '학교에도': 7, '학교는': 8, '즐겁다': 9}
encoded_list : [[2, 4, 3], [2, 5, 6, 7, 3], [8, 9]]
tensor_data : [tensor([2, 4, 3]), tensor([2, 5, 6, 7, 3]), tensor([8, 9])]

--- 결과 ---
원문: 나는 학교에 간다
Tensor: tensor([2, 4, 3])

원문: 나는 집에도 가고 학교에도 간다
Tensor: tensor([2, 5, 6, 7, 3])

원문: 학교는 즐겁다
Tensor: tensor([8, 9])



In [6]:
import torch
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

# 1~5 단계는 기존 코드와 동일
sentences = [
    "나는 학교에 간다",
    "나는 집에도 가고 학교에도 간다",
    "학교는 즐겁다"
]

tokenized_sentences = [sent.split() for sent in sentences]
tokens_all = [token for sent in tokenized_sentences for token in sent]
vocab_counts = Counter(tokens_all)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

def encode(sentence_tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in sentence_tokens]

encoded_list = [encode(sent, word_to_idx) for sent in tokenized_sentences]
print(f'word_to_idx : {word_to_idx}')
# -------------------------------------------------------------
# 6. Tensor 변환 및 패딩 (pad_sequence 활용)
# -------------------------------------------------------------
# 먼저 각 정수 리스트를 Tensor로 만듭니다.
tensor_list = [torch.tensor(seq, dtype=torch.long) for seq in encoded_list]

# pad_sequence 함수 적용
# batch_first=True : (배치 크기, 최대 문장 길이) 형태로 출력. False 이면 행렬이 전치된다.
# padding_value   : 패딩으로 채울 값 (여기서는 <PAD>의 인덱스인 0)
padded_tensors = pad_sequence(tensor_list, batch_first=True, padding_value=word_to_idx["<PAD>"])

print("--- 결과 (패딩 적용) ---")
print("최종 Tensor Shape:", padded_tensors.shape)
print(padded_tensors)

word_to_idx : {'<PAD>': 0, '<UNK>': 1, '나는': 2, '간다': 3, '학교에': 4, '집에도': 5, '가고': 6, '학교에도': 7, '학교는': 8, '즐겁다': 9}
--- 결과 (패딩 적용) ---
최종 Tensor Shape: torch.Size([3, 5])
tensor([[2, 4, 3, 0, 0],
        [2, 5, 6, 7, 3],
        [8, 9, 0, 0, 0]])


In [8]:
import torch
from collections import Counter
# from torch.nn.utils.rnn import pad_sequence

# 1~5 단계는 기존 코드와 동일
sentences = [
    "나는 학교에 간다",
    "나는 집에도 가고 학교에도 간다",
    "학교는 즐겁다"
]

tokenized_sentences = [sent.split() for sent in sentences]
tokens_all = [token for sent in tokenized_sentences for token in sent]
vocab_counts = Counter(tokens_all)

word_to_idx = {"<PAD>": 0, "<UNK>": 1}
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

def encode(sentence_tokens, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in sentence_tokens]

encoded_list = [encode(sent, word_to_idx) for sent in tokenized_sentences]

# 가장 긴 문장의 길이 계산
max_len = max(len(seq) for seq in encoded_list)
pad_idx = word_to_idx["<PAD>"]

# 최대 길이에 맞춰 0(<PAD>) 채우기
padded_encoded_list = []
for seq in encoded_list:
    # 부족한 개수만큼 pad_idx를 추가
    padded_seq = seq + [pad_idx] * (max_len - len(seq))
    padded_encoded_list.append(padded_seq)

# 한번에 Tensor로 변환
padded_tensors = torch.tensor(padded_encoded_list, dtype=torch.long)

print("--- 결과 (수동 패딩 적용) ---")
for sent, tensor in zip(sentences, padded_tensors):
    print(f"원문: {sent}")
    print(f"Padded Tensor: {tensor.tolist()}\n")

--- 결과 (수동 패딩 적용) ---
원문: 나는 학교에 간다
Padded Tensor: [2, 4, 3, 0, 0]

원문: 나는 집에도 가고 학교에도 간다
Padded Tensor: [2, 5, 6, 7, 3]

원문: 학교는 즐겁다
Padded Tensor: [8, 9, 0, 0, 0]



In [10]:
import torch
import torch.nn.functional as F

# 정수 인코딩된 단어 인덱스들 (예: ['사과', '바나나', '수박'])
targets = torch.tensor([0, 1, 3])

# 단어 집합의 총 크기 (N = 4)
vocab_size = 4

# 원-핫 인코딩 수행
one_hot_vectors = F.one_hot(targets, num_classes=vocab_size)

print(one_hot_vectors)
# 출력:
# tensor([[1, 0, 0, 0],   <- 인덱스 0 (사과)
#         [0, 1, 0, 0],   <- 인덱스 1 (바나나)
#         [0, 0, 0, 1]])  <- 인덱스 3 (수박)

tensor([[1, 0, 0, 0],
        [0, 1, 0, 0],
        [0, 0, 0, 1]])


In [18]:
import torch
import torch.nn.functional as F
from collections import Counter

# 1. 실제 입력 데이터 (문장들)
raw_sentences = [
    "나는 사과를 좋아한다",
    "나는 바나나를 좋아한다",
    "나는 사과와 바나나를 먹는다"
]

# 2. 토큰화 (공백 기준 분리)
tokenized_sentences = [sent.split() for sent in raw_sentences]

# 3. 단어 집합(Vocabulary) 생성 및 정수 인코딩 매핑
# 단어들의 전체 목록 추출
all_tokens = [token for sent in tokenized_sentences for token in sent]
print(f'all_tokens : {all_tokens}')

# 빈도수 순으로 정렬하여 단어장에 추가
vocab_counts = Counter(all_tokens)
word_to_idx = { '<PAD>': 0, '<UNK>': 1 }

for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

vocab_size = len(word_to_idx)
print(f"--- 생성된 단어 집합 (총 {vocab_size}개) ---")
print(word_to_idx)
print("-" * 40)

# 4. 문장들을 정수 리스트로 변환 (정수 인코딩)
integer_encoded_sentences = []
for sent in tokenized_sentences:
    int_seq = [word_to_idx[token] for token in sent]
    integer_encoded_sentences.append(int_seq)

# 5. 원-핫 인코딩 (One-Hot Encoding) 수행
print("--- 원-핫 인코딩 결과 ---\n")

for i, (orig_sent, int_seq) in enumerate(zip(raw_sentences, integer_encoded_sentences)):
    # 정수 리스트를 PyTorch Tensor로 변환
    tensor_seq = torch.tensor(int_seq, dtype=torch.long)
    
    # num_classes 설정: F.one_hot의 두 번째 인자(num_classes)에 전체 단어장의 크기(vocab_size)를 넣어주어야 모든 단어 벡터의 길이가 동일하게 생성됩니다.
    # PyTorch functional의 one_hot 메서드 적용 (num_classes = 전체 단어장 크기)
    one_hot_tensor = F.one_hot(tensor_seq, num_classes=vocab_size)
    
    print(f"[{i+1}번째 문장]: '{orig_sent}'")
    print(f"  └ 토큰: {tokenized_sentences[i]}")
    print(f"  └ 정수 인코딩: {int_seq}")
    print(f"  └ 원-핫 인코딩 Shape: {one_hot_tensor.shape} (단어 수: {one_hot_tensor.shape[0]}, 벡터 차원: {one_hot_tensor.shape[1]})")
    print("  └ 원-핫 벡터:")
    print(one_hot_tensor)
    print("-" * 40)

all_tokens : ['나는', '사과를', '좋아한다', '나는', '바나나를', '좋아한다', '나는', '사과와', '바나나를', '먹는다']
--- 생성된 단어 집합 (총 8개) ---
{'<PAD>': 0, '<UNK>': 1, '나는': 2, '좋아한다': 3, '바나나를': 4, '사과를': 5, '사과와': 6, '먹는다': 7}
----------------------------------------
--- 원-핫 인코딩 결과 ---

[1번째 문장]: '나는 사과를 좋아한다'
  └ 토큰: ['나는', '사과를', '좋아한다']
  └ 정수 인코딩: [2, 5, 3]
  └ 원-핫 인코딩 Shape: torch.Size([3, 8]) (단어 수: 3, 벡터 차원: 8)
  └ 원-핫 벡터:
tensor([[0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0]])
----------------------------------------
[2번째 문장]: '나는 바나나를 좋아한다'
  └ 토큰: ['나는', '바나나를', '좋아한다']
  └ 정수 인코딩: [2, 4, 3]
  └ 원-핫 인코딩 Shape: torch.Size([3, 8]) (단어 수: 3, 벡터 차원: 8)
  └ 원-핫 벡터:
tensor([[0, 0, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 1, 0, 0, 0],
        [0, 0, 0, 1, 0, 0, 0, 0]])
----------------------------------------
[3번째 문장]: '나는 사과와 바나나를 먹는다'
  └ 토큰: ['나는', '사과와', '바나나를', '먹는다']
  └ 정수 인코딩: [2, 6, 4, 7]
  └ 원-핫 인코딩 Shape: torch.Size([4, 8]) (단어 수: 4, 벡터 차원: 8)
  └ 원-핫 벡터:
t

In [24]:
import torch
import torch.nn.functional as F
from collections import Counter
from torch.nn.utils.rnn import pad_sequence

# 1. 실제 입력 데이터 (문장들)
raw_sentences = [
    "나는 사과를 좋아한다",                # 3개 단어
    "나는 바나나를 좋아한다",                # 3개 단어
    "나는 사과와 바나나를 먹는다"           # 4개 단어
]

# 2. 토큰화 (공백 기준 분리)
tokenized_sentences = [sent.split() for sent in raw_sentences]

# 3. 단어 집합(Vocabulary) 생성
# 0번 인덱스는 패딩(PAD) 토큰으로 예약해 둡니다.
all_tokens = [token for sent in tokenized_sentences for token in sent]
vocab_counts = Counter(all_tokens)

word_to_idx = { "<PAD>": 0, "<UNK>": 1 }  # 패딩용 토큰 추가
for word, _ in vocab_counts.most_common():
    if word not in word_to_idx:
        word_to_idx[word] = len(word_to_idx)

vocab_size = len(word_to_idx)
print(vocab_size)

# 4. 각 문장을 정수 Tensor 리스트로 변환
tensor_list = [
    torch.tensor([word_to_idx.get(token, word_to_idx["<UNK>"]) for token in sent], dtype=torch.long) for sent in tokenized_sentences
]
print(f'tensor_list : {tensor_list}')

# 5. 패딩(Padding) 적용하여 길이가 다른 문장들을 정방형 Tensor로 맞춤 (batch_first=True)
# 가장 긴 문장(4개 단어)에 맞춰 짧은 문장 뒤에 0(<PAD>)을 채워 넣습니다.
all_tensor_seq = pad_sequence(tensor_list, batch_first=True, padding_value=0)
print(f'all_tensor_seq : {all_tensor_seq}')

# 6. 전체 문장 일괄 원-핫 인코딩 수행
one_hot_tensor = F.one_hot(all_tensor_seq, num_classes=vocab_size)

# 7. 통합 출력
print(f"--- 생성된 단어 집합 (총 {vocab_size}개, <PAD> 포함) ---")
print(word_to_idx)
print("\n--- 패딩 처리된 정수 Tensor (Shape: [3, 4]) ---")
print(all_tensor_seq)

print("\n--- 전체 문장 원-핫 인코딩 결과 ---")
print(f"Tensor Shape: {one_hot_tensor.shape} (문장 수: {one_hot_tensor.shape[0]}, 문장당 최대 단어 수: {one_hot_tensor.shape[1]}, 단어장 크기: {one_hot_tensor.shape[2]})\n")
print(one_hot_tensor)

8
tensor_list : [tensor([2, 5, 3]), tensor([2, 4, 3]), tensor([2, 6, 4, 7])]
all_tensor_seq : tensor([[2, 5, 3, 0],
        [2, 4, 3, 0],
        [2, 6, 4, 7]])
--- 생성된 단어 집합 (총 8개, <PAD> 포함) ---
{'<PAD>': 0, '<UNK>': 1, '나는': 2, '좋아한다': 3, '바나나를': 4, '사과를': 5, '사과와': 6, '먹는다': 7}

--- 패딩 처리된 정수 Tensor (Shape: [3, 4]) ---
tensor([[2, 5, 3, 0],
        [2, 4, 3, 0],
        [2, 6, 4, 7]])

--- 전체 문장 원-핫 인코딩 결과 ---
Tensor Shape: torch.Size([3, 4, 8]) (문장 수: 3, 문장당 최대 단어 수: 4, 단어장 크기: 8)

tensor([[[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 1, 0],
         [0, 0, 0, 0, 1, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]]])


In [32]:
from torch.nn.utils.rnn import pad_sequence

new_raw_sentences = [
    "너는 복숭아를 좋아한다",
    "나는 사과를 아주 잘 먹는다"
]

# 토큰화 작업 (공백 기준 분리)
new_tokenized_sentences = [sen.split() for sen in new_raw_sentences]
print(new_tokenized_sentences)

# 정수 인코딩
new_tensor_list = [
    torch.tensor([word_to_idx.get(token, word_to_idx['<UNK>'])  for token in sen], dtype=torch.long) for sen in new_tokenized_sentences 
]
print(new_tensor_list)

# 패딩 처리
new_all_tensor_seq = pad_sequence(new_tensor_list, batch_first=True, padding_value=word_to_idx['<PAD>'])
print(f'new_all_tensor_seq :\n{new_all_tensor_seq}')

# 원-핫 인코딩
new_one_hot_tensor = F.one_hot(new_all_tensor_seq, num_classes=vocab_size)
print(f'new_one_hot_tensor: {new_one_hot_tensor} ')

# 결과 출력
print(f'new_one_hot_tensor: {new_one_hot_tensor} ')

[['너는', '복숭아를', '좋아한다'], ['나는', '사과를', '아주', '잘', '먹는다']]
[tensor([1, 1, 3]), tensor([2, 5, 1, 1, 7])]
new_all_tensor_seq :
tensor([[1, 1, 3, 0, 0],
        [2, 5, 1, 1, 7]])
new_one_hot_tensor: tensor([[[0, 1, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]]]) 
new_one_hot_tensor: tensor([[[0, 1, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 1, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0],
         [1, 0, 0, 0, 0, 0, 0, 0]],

        [[0, 0, 1, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 1, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 1, 0, 0, 0, 0, 0, 0],
         [0, 0, 0, 0, 0, 0, 0, 1]]]) 


In [34]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "나는 사과를 좋아하고 사과를 먹는다",
    "나는 바나나를 좋아한다",
    "나는 사과와 바나나를 모두 먹는다"
]

# TF-IDF 객체 생성 및 학습
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(corpus)
print(tfidf_matrix)

print("단어 집합 (Vocabulary):")
print(tfidf_vectorizer.vocabulary_)

print("\nTF-IDF Matrix (Dense Array):")
# 단순 빈도(정수)가 아닌 연속형 실숫값 가중치로 변환됨
print(tfidf_matrix.toarray().round(2))

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 12 stored elements and shape (3, 8)>
  Coords	Values
  (0, 0)	0.24259369753845733
  (0, 4)	0.8214936700177023
  (0, 6)	0.4107468350088512
  (0, 1)	0.31238355521006117
  (1, 0)	0.4254405389711991
  (1, 3)	0.5478321549274363
  (1, 7)	0.7203334490549893
  (2, 0)	0.3154441510317797
  (2, 1)	0.4061917781433946
  (2, 3)	0.4061917781433946
  (2, 5)	0.5340933749435833
  (2, 2)	0.5340933749435833
단어 집합 (Vocabulary):
{'나는': 0, '사과를': 4, '좋아하고': 6, '먹는다': 1, '바나나를': 3, '좋아한다': 7, '사과와': 5, '모두': 2}

TF-IDF Matrix (Dense Array):
[[0.24 0.31 0.   0.   0.82 0.   0.41 0.  ]
 [0.43 0.   0.   0.55 0.   0.   0.   0.72]
 [0.32 0.41 0.53 0.41 0.   0.53 0.   0.  ]]


In [35]:
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity

# 1. 비교할 문장 데이터 준비
corpus = [
    "나는 사과를 좋아하고 사과를 먹는다",   # 문장 0
    "나는 바나나를 좋아한다",            # 문장 1
    "나는 사과와 바나나를 모두 먹는다"     # 문장 2
]

# 2. scikit-learn을 이용한 TF-IDF 벡터화
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)

# TF-IDF 결과를 NumPy 밀집 배열(Dense Array)로 변환
tfidf_dense = tfidf_matrix.toarray()

print("--- TF-IDF Matrix Shape ---")
print(tfidf_dense.shape) # (3개의 문장, 단어장 크기)
print("\n--- 단어 집합 (Vocabulary) ---")
print(vectorizer.vocabulary_)

# =====================================================================
# 방식 1: PyTorch를 이용한 코사인 유사도 계산
# =====================================================================
# NumPy 배열을 PyTorch FloatTensor로 변환
tfidf_tensor = torch.tensor(tfidf_dense, dtype=torch.float32)

# torch.nn.functional.cosine_similarity를 활용한 모든 문장 쌍(Pair) 간 유사도 계산
# dim=-1을 기준으로 벡터 내적 및 L2 노름을 계산합니다.
num_sentences = tfidf_tensor.size(0)
pytorch_similarity_matrix = torch.zeros((num_sentences, num_sentences))

for i in range(num_sentences):
    for j in range(num_sentences):
        # unsqueeze(0)으로 (1, D) 차원을 맞춰준 뒤 계산
        sim = F.cosine_similarity(tfidf_tensor[i].unsqueeze(0), tfidf_tensor[j].unsqueeze(0))
        pytorch_similarity_matrix[i][j] = sim.item()

print("\n==========================================")
print("[방식 1] PyTorch 계산 코사인 유사도 행렬:")
print(pytorch_similarity_matrix.numpy().round(4))

# =====================================================================
# 방식 2: scikit-learn을 이용한 코사인 유사도 계산 (검증용)
# =====================================================================
sklearn_similarity_matrix = sklearn_cosine_similarity(tfidf_matrix)

print("\n[방식 2] scikit-learn 계산 코사인 유사도 행렬:")
print(sklearn_similarity_matrix.round(4))

# =====================================================================
# 결과 해석
# =====================================================================
print("\n--- 문장 간 유사도 비교 결과 ---")
print(f"문장 0 ('사과를 좋아하고...') vs 문장 1 ('바나나를 좋아한다...'): {pytorch_similarity_matrix[0][1]:.4f}")
print(f"문장 0 ('사과를 좋아하고...') vs 문장 2 ('사과와 바나나를...'):   {pytorch_similarity_matrix[0][2]:.4f}")
print(f"문장 1 ('바나나를 좋아한다') vs 문장 2 ('사과와 바나나를...'):   {pytorch_similarity_matrix[1][2]:.4f}")

--- TF-IDF Matrix Shape ---
(3, 8)

--- 단어 집합 (Vocabulary) ---
{'나는': 0, '사과를': 4, '좋아하고': 6, '먹는다': 1, '바나나를': 3, '좋아한다': 7, '사과와': 5, '모두': 2}

[방식 1] PyTorch 계산 코사인 유사도 행렬:
[[1.     0.1032 0.2034]
 [0.1032 1.     0.3567]
 [0.2034 0.3567 1.    ]]

[방식 2] scikit-learn 계산 코사인 유사도 행렬:
[[1.     0.1032 0.2034]
 [0.1032 1.     0.3567]
 [0.2034 0.3567 1.    ]]

--- 문장 간 유사도 비교 결과 ---
문장 0 ('사과를 좋아하고...') vs 문장 1 ('바나나를 좋아한다...'): 0.1032
문장 0 ('사과를 좋아하고...') vs 문장 2 ('사과와 바나나를...'):   0.2034
문장 1 ('바나나를 좋아한다') vs 문장 2 ('사과와 바나나를...'):   0.3567


In [36]:
import torch
import torch.nn as nn

# --- 이전 단계에서 완성된 데이터 재사용 ---
# 단어장 크기 (PAD 포함 8개): 0:<PAD>, 1:나는, 2:좋아한다, 3:바나나를, 4:사과를, 5:먹는다, 6:사과와, 7:모두
vocab_size = 8

# 패딩(PAD=0)이 채워진 정수 인코딩 문장 Tensor (Shape: [3, 4])
# 문장 1: 나는 사과를 좋아한다 <PAD>
# 문장 2: 나는 바나나를 좋아한다 <PAD>
# 문장 3: 나는 사과와 바나나를 먹는다
all_tensor_seq = torch.tensor([
    [1, 4, 2, 0],
    [1, 3, 2, 0],
    [1, 6, 3, 5]
], dtype=torch.long)

# 1. nn.Embedding 레이어 정의
embedding_dim = 4  # 임베딩 벡터의 차원 수 (원하는 임의의 크기로 지정 가능)

# padding_idx=0으로 지정하면 0번(<PAD>) 토큰의 임베딩 벡터는 항상 0으로 유지되고 학습되지 않습니다.
embedding_layer = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim,
    padding_idx=0
)

# 2. 정수 인코딩 Tensor를 임베딩 레이어에 전달 (룩업 테이블 연산)
dense_embedded_tensor = embedding_layer(all_tensor_seq)

# 3. 결과 출력
print(f"--- 입력 정수 Tensor Shape: {all_tensor_seq.shape} ---")
print(all_tensor_seq)

print(f"\n--- nn.Embedding 출력 Tensor Shape: {dense_embedded_tensor.shape} ---")
print(f"(배치 크기: {dense_embedded_tensor.shape[0]}, 문장 길이: {dense_embedded_tensor.shape[1]}, 임베딩 차원: {dense_embedded_tensor.shape[2]})\n")

print("--- 밀집 벡터 변환 결과 (일부) ---")
print(f"문장 1의 첫 단어('나는', idx=1)의 밀집 벡터:\n{dense_embedded_tensor[0, 0]}")
print(f"\n문장 1의 패딩('<PAD>', idx=0)의 밀집 벡터 (0으로 고정):\n{dense_embedded_tensor[0, 3]}")

--- 입력 정수 Tensor Shape: torch.Size([3, 4]) ---
tensor([[1, 4, 2, 0],
        [1, 3, 2, 0],
        [1, 6, 3, 5]])

--- nn.Embedding 출력 Tensor Shape: torch.Size([3, 4, 4]) ---
(배치 크기: 3, 문장 길이: 4, 임베딩 차원: 4)

--- 밀집 벡터 변환 결과 (일부) ---
문장 1의 첫 단어('나는', idx=1)의 밀집 벡터:
tensor([ 0.1645, -0.9611, -0.7958,  0.5703], grad_fn=<SelectBackward0>)

문장 1의 패딩('<PAD>', idx=0)의 밀집 벡터 (0으로 고정):
tensor([0., 0., 0., 0.], grad_fn=<SelectBackward0>)
